In [ ]:
# ============================================================
# 第 5 章 Part-2：LoRA 简介 + 训练/测试集切分 + 用 litgpt 做 LoRA 指令微调
# ------------------------------------------------------------
# 流程：加载数据 -> 85/15 切分并存成 json -> 用 litgpt finetune_lora 微调 phi-2
#       -> 分别用「基座模型」和「微调后模型」对测试集生成回答，便于后续对比。
# ============================================================
#Introduction to LoRA Creating training and test sets

In [ ]:
# 目的：重新加载与 Part-1 相同的指令数据集。
import json


file_path = "LLM-workshop-2024/06_finetuning/instruction-data.json"

with open(file_path, "r") as file:
    data = json.load(file)
print("Number of entries:", len(data))

In [ ]:
# 切分数据集：85% 训练，15% 测试。
# 注意这里是「按顺序切片」而非随机打乱；教学场景够用，实际项目通常应先 shuffle 以避免顺序偏差。
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.15)    # 15% for testing

# 前 85% 作训练集，其余作测试集。（test_portion 变量在此并未真正用于切片，仅作展示）
train_data = data[:train_portion]
test_data = data[train_portion:]

In [ ]:
# 打印两个子集大小，并把它们分别落盘为 train.json / test.json。
# 存成文件的原因：下一步的 litgpt 命令行工具是独立进程，需要从磁盘读取训练数据。
print("Training set length:", len(train_data))
print("Test set length:", len(test_data))
# indent=4 让 JSON 可读；train.json 将作为 LoRA 微调的输入数据。
with open("train.json", "w") as json_file:
    json.dump(train_data, json_file, indent=4)

with open("test.json", "w") as json_file:
    json.dump(test_data, json_file, indent=4)

In [ ]:
# --------- 指令微调阶段 ---------
#Instruction finetuning

In [ ]:
# 用 litgpt 的 LoRA 微调命令对 microsoft/phi-2 做指令微调（! 前缀表示在 shell 中执行）。
#
# LoRA (Low-Rank Adaptation) 原理：
#   大模型全量微调需要更新所有权重 W（数十亿参数），显存/存储代价极高。
#   LoRA 冻结原始权重 W 不动，只在旁路上新增一对低秩矩阵 A(d×r) 和 B(r×k)，
#   用 W + B·A 近似微调后的权重（其中秩 r << d,k，参数量极小，通常只占原模型的百分之几）。
#   训练时只更新 A、B，因此显存占用、可训练参数量、存储的适配器体积都大幅下降，
#   且可为不同任务保存不同的小适配器，随时叠加到同一个基座上。
#
# 各参数含义：
#   microsoft/phi-2                  ：作为基座 (base model) 的预训练模型
#   --data JSON                      ：使用 JSON 数据加载器
#   --data.val_split_fraction 0.1    ：从训练数据里再切 10% 作验证集（monitor 过拟合）
#   --data.json_path train.json      ：训练数据文件路径
#   --train.epochs 3                 ：训练 3 个 epoch（整份数据过 3 遍）
#   --train.log_interval 100         ：每 100 个训练 step 打印一次日志
!litgpt finetune_lora microsoft/phi-2 \
--data JSON \
--data.val_split_fraction 0.1 \
--data.json_path train.json \
--train.epochs 3 \
--train.log_interval 100

In [ ]:
# --------- 用「基座模型」对测试集生成回答（作为微调前的对照基线）---------
#Generate and save the test set model responses of the base model

In [ ]:
# 重新定义 format_input（与 Part-1 相同），用于把测试样本包装成统一提示模板。
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

# 打印测试集第 0 条格式化后的提示，确认模板正确。
print(format_input(test_data[0]))

In [ ]:
# 加载「未微调的基座模型」phi-2，用于生成微调前的对照回答。
from litgpt import LLM

llm = LLM.load("microsoft/phi-2")

In [ ]:
# 遍历整个测试集，用基座模型生成回答，结果写回每条样本的 "base_model" 字段。
from tqdm import tqdm

for i in tqdm(range(len(test_data))):
    # ✅ 已修复：原代码直接把整条 dict test_data[i] 传给 generate（模型只会收到 dict 的字符串形式）；
    #    现改为先用 format_input(...) 包装成标准指令提示再生成。
    response = llm.generate(format_input(test_data[i]))
    test_data[i]["base_model"] = response

In [ ]:
# 查看第 1 条样本，确认已新增 base_model 字段（基座模型的回答）。
test_data[1]

In [ ]:
# --------- 用「微调后模型」对测试集生成回答 ---------
#Generate and save the test set model responses of the finetuned model

In [ ]:
# 释放基座模型显存后，加载 LoRA 微调产出的最终模型。
# out/finetune/lora/final/ 是上面 litgpt finetune_lora 命令默认的输出目录，
# litgpt 在此已把 LoRA 适配器合并进基座权重，可直接作为完整模型加载。
from litgpt import LLM

del llm  # 删除基座模型引用，释放 GPU 显存，避免同时加载两个模型爆显存
llm2 = LLM.load("out/finetune/lora/final/")

In [ ]:
# 用微调后模型再次遍历测试集，回答写入 "finetuned_model" 字段，
# 与前面的 base_model 字段并列，方便对比「微调前 vs 微调后」的效果。
from tqdm import tqdm

for i in tqdm(range(len(test_data))):
    # ✅ 已修复：同上，改为传入 format_input(test_data[i]) 而非整条 dict。
    response = llm2.generate(format_input(test_data[i]))
    test_data[i]["finetuned_model"] = response